In [ ]:
#libraries
import pandas as pd
import numpy as np

DATA CLEANING AND STUFF

In [ ]:
#load the data
pipes = pd.read_csv("Water_Pipes_Final_csv.csv", encoding='cp1252', dtype={10: str})
b1 = pd.read_csv("Main_Breaks_Post_March_2022.csv")
b2 = pd.read_csv("Main_Breaks_Jan_2013_to_March_2022.csv")
#b3 = pd.read_csv("Main_Breaks_Pre_2013.csv") #not going to use this for today bc its too different and need more time

In [ ]:
#need to fix some attribute names to stay consistent
b1_clean = b1.rename(columns={
    "entityuid": "pipe_tag",
    "initiateda": "break_date"
})[["pipe_tag", "break_date"]]

b2_clean = b2.rename(columns={
    "pipe_tag": "pipe_tag",
    "date_repor": "break_date"
})[["pipe_tag", "break_date"]]

In [ ]:
#fix dates
b1_clean["break_date"] = pd.to_datetime(b1_clean["break_date"], errors="coerce")
b2_clean["break_date"] = pd.to_datetime(b2_clean["break_date"], errors="coerce")

In [ ]:
b1_clean.isna().sum()

,0
pipe_tag,0
break_date,0


In [ ]:
b2_clean.isna().sum()

,0
pipe_tag,0
break_date,0


In [ ]:
breaks = pd.concat([b1_clean, b2_clean], ignore_index=True)

In [ ]:
breaks.head()

,pipe_tag,break_date
0,wp526,2022-08-10
1,p3844,2022-08-12
2,p3978,2022-08-19
3,wp37055,2022-08-22
4,wp15798,2022-08-22


In [ ]:
#now time to clean pipes

pipes_clean = pipes.copy()
pipes_clean = pipes_clean[pipes_clean["cbu"].str.lower() == "y"]
pipes_clean = pipes_clean[pipes_clean["diameter"] > 0]
pipes_clean = pipes_clean[[
    "OBJECTID", "tag", "diameter", "length", "material", "yr_inst"  # attributes i'm using for this first model
]]
pipes_clean["yr_inst"] = pd.to_numeric(pipes_clean["yr_inst"], errors='coerce')
pipes_clean = pipes_clean.dropna(subset=["yr_inst"])
pipes_clean["yr_inst"] = pipes_clean["yr_inst"].astype(int)

In [ ]:
pipes_clean.head()

,OBJECTID,tag,diameter,length,material,yr_inst
27,28,k4014,6.0,5,CI,1973
30,31,m411,6.0,425,CI,1961
31,32,m4117,2.0,58,C,1983
32,33,m4130,12.0,28,DI,1987
33,34,m4129,12.0,4,DI,1987


In [ ]:
breaks = breaks.merge(
    pipes[["tag"]],
    left_on="pipe_tag",
    right_on="tag",
    how="inner"
)

In [ ]:
snapshots = pd.date_range("2012-01-01", "2026-01-01", freq="YS")
rows = []

for _, pipe in pipes_clean.iterrows():
    tag = pipe["tag"]
    install_year = pipe["yr_inst"]

    pipe_breaks = breaks.loc[breaks["pipe_tag"] == tag, "break_date"].sort_values()

    for snap in snapshots:
        if snap.year < install_year:
            continue

        past = pipe_breaks[pipe_breaks < snap]

        rows.append({
            "tag": tag,
            "snapshot": snap,
            "diameter": pipe["diameter"],
            "length": pipe["length"],
            "material": pipe["material"],
            "age_years": snap.year - install_year,
            "breaks_so_far": len(past),
            "months_since_last":
                (snap - past.max()).days / 30 if len(past) > 0 else np.nan
        })

snap_df = pd.DataFrame(rows)

In [ ]:
snap_df.head(100)

,tag,snapshot,diameter,length,material,age_years,breaks_so_far,months_since_last
0,k4014,2012-01-01,6.0,5,CI,39,0,NaN
1,k4014,2013-01-01,6.0,5,CI,40,0,NaN
2,k4014,2014-01-01,6.0,5,CI,41,0,NaN
3,k4014,2015-01-01,6.0,5,CI,42,0,NaN
4,k4014,2016-01-01,6.0,5,CI,43,0,NaN
...,...,...,...,...,...,...,...,...
95,m4032,2017-01-01,6.0,18,CI,52,0,NaN
96,m4032,2018-01-01,6.0,18,CI,53,0,NaN
97,m4032,2019-01-01,6.0,18,CI,54,0,NaN
98,m4032,2020-01-01,6.0,18,CI,55,0,NaN


In [ ]:
#make it a csv
snap_df.to_csv("snapshots.csv", index=False)

THE ACTUAL MACHINE LEARNING PART:

In [ ]:
X = 12 #months

In [ ]:
def broke_in_next_X(tag, snap, months):
    future = breaks[
        (breaks["pipe_tag"] == tag) &
        (breaks["break_date"] > snap) &
        (breaks["break_date"] <= snap + pd.DateOffset(months=months))
    ]
    return int(len(future) > 0)

snap_df["Y"] = snap_df.apply(
    lambda r: broke_in_next_X(r["tag"], r["snapshot"], X),
    axis=1
)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, mean_absolute_error, recall_score
from xgboost import XGBClassifier

In [ ]:
num_features = ["diameter", "length", "age_years", "breaks_so_far", "months_since_last"]
cat_features = ["material"]
Xmat = snap_df[num_features + cat_features]
y = snap_df["Y"]

train = snap_df["snapshot"] < "2022-01-01"
X_train = Xmat[train]
X_test  = Xmat[~train]
y_train = y[train]
y_test  = y[~train]

In [ ]:
#in case of missing values
num_imp = SimpleImputer(strategy="median")
X_train_num = num_imp.fit_transform(X_train[num_features])
X_test_num  = num_imp.transform(X_test[num_features])

#one hot encoding
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat = ohe.fit_transform(X_train[cat_features])
X_test_cat  = ohe.transform(X_test[cat_features])

#train test sets
X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final  = np.hstack([X_test_num,  X_test_cat])

MODEL 1: LOGISTIC REGRESSION W/ LASSO

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled  = scaler.transform(X_test_final)

lasso_logit = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    class_weight="balanced",
    max_iter=1000
)

In [ ]:
lasso_logit.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, penalty='l1',
                   solver='liblinear')

In [ ]:
p_test_lasso = lasso_logit.predict_proba(X_test_scaled)[:, 1]

MODEL 2: XGBOOST

In [ ]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False
)

In [ ]:
xgb.fit(X_train_final, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [19:35:24] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
p_test_xgb = xgb.predict_proba(X_test_final)[:, 1]

EVALUATION

In [ ]:
print("LASSO ROC AUC:", roc_auc_score(y_test, p_test_lasso))
print("LASSO PR AUC :", average_precision_score(y_test, p_test_lasso))
print("LASSO Log Loss:", log_loss(y_test, p_test_lasso))
print("LASSO MAE :", mean_absolute_error(y_test, p_test_lasso))
print("-------------------------------------")
print("XGB ROC AUC:", roc_auc_score(y_test, p_test_xgb))
print("XGB PR AUC :", average_precision_score(y_test, p_test_xgb))
print("XGB Log Loss:", log_loss(y_test, p_test_xgb))
print("XGB MAE :", mean_absolute_error(y_test, p_test_xgb))

#ROC AUC = Probability that a randomly chosen broken pipe gets a higher risk score than a randomly chosen non-broken pipe
#Precision = How well the model identifies actual breaks among high-risk predictions (not good right now)

LASSO ROC AUC: 0.8712571421832893
LASSO PR AUC : 0.022089798766199983
LASSO Log Loss: 0.4764118945593436
LASSO MAE : 0.29070558216594583
-------------------------------------
XGB ROC AUC: 0.8504290807200374
XGB PR AUC : 0.01978163502033787
XGB Log Loss: 0.012905481319937706
XGB MAE : 0.003543555038049817


PREDICTION

In [ ]:
today = pd.Timestamp.today().normalize()
rows_today = []

for _, pipe in pipes_clean.iterrows():
    tag = pipe["tag"]
    install_year = pipe["yr_inst"]

    if today.year < install_year:
        continue

    pipe_breaks = breaks.loc[breaks["pipe_tag"] == tag, "break_date"]

    past = pipe_breaks[pipe_breaks < today]

    rows_today.append({
        "tag": tag,
        "diameter": pipe["diameter"],
        "length": pipe["length"],
        "material": pipe["material"],
        "age_years": today.year - install_year,
        "breaks_so_far": len(past),
        "months_since_last":
            (today - past.max()).days / 30 if len(past) > 0 else np.nan
    })

today_df = pd.DataFrame(rows_today)

In [ ]:
X_today_num = num_imp.transform(today_df[num_features])
X_today_cat = ohe.transform(today_df[cat_features])
X_today_final = np.hstack([X_today_num, X_today_cat])
X_today_scaled = scaler.transform(X_today_final)

In [ ]:
today_df["prob_lasso"] = lasso_logit.predict_proba(X_today_scaled)[:, 1]
today_df["prob_xgb"] = xgb.predict_proba(X_today_final)[:, 1]

In [ ]:
#highest risk pipes lasso
today_df.sort_values("prob_lasso", ascending=False).head(20)

,tag,diameter,length,material,age_years,breaks_so_far,months_since_last,prob_lasso,prob_xgb
10835,q5311,10.0,3149,CI,61,0,NaN,1.000000,0.009386
21078,wp25745,8.0,1388,CI,61,5,74.466667,1.000000,0.034180
1171,l45120,6.0,2468,CI,70,0,NaN,1.000000,0.014211
18531,p331,6.0,2298,DI,47,0,NaN,1.000000,0.063202
7933,v293,6.0,1734,DI,41,2,77.366667,1.000000,0.056707
21102,wp33944,12.0,2237,DI,8,0,NaN,1.000000,0.000386
182,wp2595,12.0,1168,DI,41,4,7.000000,1.000000,0.007889
3864,p504,18.0,1359,CI,63,3,19.400000,1.000000,0.001185
14550,q3954,4.0,909,CI,61,4,48.266667,0.999999,0.024365
5190,wp14804,6.0,1746,CI,59,0,NaN,0.999997,0.035719


In [ ]:
#highest risk pipes xgboost
today_df.sort_values("prob_xgb", ascending=False).head(20)

,tag,diameter,length,material,age_years,breaks_so_far,months_since_last,prob_lasso,prob_xgb
22028,s40105,8.0,599,CI,58,2,83.566667,0.999363,0.462861
14841,s46260,12.0,294,CI,67,2,61.666667,0.992202,0.342517
10175,wp29262,6.0,1178,CI,54,2,19.600000,0.999995,0.315873
5731,wp18401,4.0,216,CI,1831,2,72.800000,0.991885,0.286979
28205,wp27467,6.0,894,CI,57,2,38.166667,0.999954,0.263330
26975,q5173,6.0,413,CI,67,2,75.233333,0.997839,0.241826
19101,wp10211,6.0,349,CI,2026,3,92.833333,0.999425,0.236675
16492,wp16053,6.0,262,DI,38,2,72.400000,0.984032,0.230132
11798,wp8765,6.0,141,DI,38,1,83.133333,0.776785,0.224811
7237,r38154,6.0,263,DI,38,1,92.900000,0.893361,0.208876


In [ ]:
#make it a csv
output = today_df[[
    "tag",
    "prob_lasso",
    "prob_xgb"
]].sort_values("prob_xgb", ascending=False)

output.to_csv("pipe_break_risk_today.csv", index=False)